In [18]:
import os
import pandas as pd
import numpy as np

In [19]:
def transformar_con_ancla(ruta_archivo):
    try:
        # 1. Carga bruta inicial (sin asumir encabezados) y recorte hasta BH (60 columnas)
        df_raw = pd.read_excel(ruta_archivo, sheet_name=0, header=None)
        df_raw = df_raw.iloc[:, :60] 

        # 2. Buscar el Ancla Dinámica (Fila donde empiezan los encabezados reales)
        fila_ancla = -1
        for i in range(15): # Escanea las primeras 15 filas
            fila_texto = df_raw.iloc[i].fillna("").astype(str).str.upper().tolist()
            if any("UNIDAD" in celda for celda in fila_texto) or any("RG" in celda for celda in fila_texto):
                fila_ancla = i
                break
                
        if fila_ancla == -1:
            print(f"⚠️ No se encontró la cabecera (UNIDAD o RG) en {os.path.basename(ruta_archivo)}")
            return None

        # 3. Extraer los 3 niveles de títulos a partir del ancla
        fila_sup = df_raw.iloc[fila_ancla].fillna("").astype(str).tolist()
        fila_med = df_raw.iloc[fila_ancla + 1].fillna("").astype(str).tolist()
        fila_inf = df_raw.iloc[fila_ancla + 2].fillna("").astype(str).tolist()

        # 4. Construcción lógica de nombres (aplanado de celdas combinadas)
        nuevos_nombres = []
        sup_actual, med_actual = "", ""
        
        for i in range(len(df_raw.columns)):
            val_sup = fila_sup[i].replace("nan", "").strip()
            val_med = fila_med[i].replace("nan", "").strip()
            val_inf = fila_inf[i].replace("nan", "").strip()
            
            # Arrastrar valores si hay celdas combinadas arriba
            if val_sup and "Unnamed" not in val_sup: sup_actual = val_sup
            if val_med and "Unnamed" not in val_med: med_actual = val_med
                
            # Filtro anti-basura: Ignorar títulos institucionales gigantes (> 80 caracteres)
            clean_sup = sup_actual if len(sup_actual) < 80 else f"BLOQUE_{i}"
            clean_med = med_actual if len(med_actual) < 80 else f"SUB_{i}"
            clean_inf = val_inf if len(val_inf) < 80 else ""
            
            # Las primeras 5 columnas (identificadores) no necesitan fusionar 3 niveles
            if i < 5:
                partes = [clean_sup]
            else:
                partes = [p for p in [clean_sup, clean_med, clean_inf] if p]
                
            nombre_final = "_".join(partes).replace("\n", " ").strip("_")
            nuevos_nombres.append(nombre_final if nombre_final else f"COL_{i}")

        # 5. Garantizar nombres únicos (Previene el InvalidIndexError)
        nombres_unicos = []
        conteo = {}
        for nom in nuevos_nombres:
            if nom not in conteo:
                nombres_unicos.append(nom)
                conteo[nom] = 1
            else:
                nombres_unicos.append(f"{nom}_{conteo[nom]}")
                conteo[nom] += 1

        # 6. Extracción de los datos reales (Empiezan 3 filas debajo del ancla)
        df_datos = df_raw.iloc[fila_ancla + 3:].copy()
        df_datos.columns = nombres_unicos

        # 7. Limpieza y relleno de celdas combinadas (ffill) para las primeras 5 columnas
        cols_id = df_datos.columns[:5]
        df_datos[cols_id] = df_datos[cols_id].replace(["", "None", "nan", "NaN"], np.nan).infer_objects(copy=False).ffill()

        # Eliminar filas totalmente vacías y registrar el origen
        df_datos = df_datos.dropna(how='all').reset_index(drop=True)
        df_datos['ARCHIVO_ORIGEN'] = os.path.basename(ruta_archivo)
        
        return df_datos

    except Exception as e:
        print(f"❌ Error al transformar {os.path.basename(ruta_archivo)}: {e}")
        return None

In [20]:
#Función para reparar nombres repetidos en un DataFrame
def reparar_columnas_duplicadas(df):
    nombres_vistos = {}
    nuevos_nombres = []
    
    for col in df.columns:
        # Convertimos a string por seguridad
        nombre_str = str(col) 
        
        if nombre_str not in nombres_vistos:
            nombres_vistos[nombre_str] = 0
            nuevos_nombres.append(nombre_str)
        else:
            nombres_vistos[nombre_str] += 1
            # Le agregamos el sufijo numérico para hacerlo único
            nombre_unico = f"{nombre_str}_{nombres_vistos[nombre_str]}"
            nuevos_nombres.append(nombre_unico)
            
    df.columns = nuevos_nombres
    return df

In [21]:
# Definimos la ruta exacta sincronizada de OneDrive a ruta local
ruta_base = r'C:\Users\inter\Universidad Externado de Colombia\José William Cifuentes Sánchez - bases'

#Manejo de pats y librerías para acceder a la carpeta compartida de OneDrive
try:
    os.chdir(ruta_base)
    print(f"Directorio actual: {os.getcwd()}")
    
    # Listado de archivos en la carpeta compartida
    archivos = os.listdir('.')
    print("\nArchivos disponibles en la carpeta compartida:")
    for archivo in archivos:
        print(f"- {archivo}")        
except FileNotFoundError:
    print("Error: No se encontró la ruta. Asegúrate de que OneDrive esté iniciado.")

Directorio actual: C:\Users\inter\Universidad Externado de Colombia\José William Cifuentes Sánchez - bases

Archivos disponibles en la carpeta compartida:
- .849C9593-D756-4E56-8D6E-42412F2A707B
- SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 23022025 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 28022025 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 29092024 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 31082024 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA 02032025 PT. DÁVILA.xlsx
- SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx
- SALA DE DETENIDOS DÍA 12032025 PT. DÁVILA.xlsx
- SALA DE DETENIDOS DÍA 17032025 IT. ACOSTA.xlsx
- SALA DE DETENIDOS DÍA 18092024 IT. ACOSTA.xlsx
- SALA DE DETENIDOS DÍA 20042025 PT.DÁVILA.xlsx
- SALA DE DETENIDOS DÍA 22042025 IT. ACOSTA.xlsx
- SALA DE DETENIDOS DÍA 27022024 IT. TOBÓN.xlsx
- SALA DE DETENIDOS DÍA 30062024 IT. TOBÓN.xlsx
- SALA DE DETENIDOS DÍA 30092024 SI. GARCIA.xlsx
- SALA DE DETENIDOS

In [22]:
# --- BUCLE DE PROCESAMIENTO ---

ruta_base = r'C:\Users\inter\Universidad Externado de Colombia\José William Cifuentes Sánchez - bases'
archivos = [f for f in os.listdir(ruta_base) if f.endswith(('.xlsx', '.xls'))]

todos_los_datos = []

for archivo in archivos:
    ruta_full = os.path.join(ruta_base, archivo)
    print(f"📖 Leyendo primera hoja de: {archivo}")
    
    df_temp = transformar_con_ancla(ruta_full)
    if df_temp is not None:
        todos_los_datos.append(df_temp)

# Resultado final
if todos_los_datos:
    print(f"\n✅ Total de archivos procesados: {len(todos_los_datos)}")
    # Mostrar las columnas del primero para verificar los prefijos
    print("Columnas detectadas:", todos_los_datos[0].columns.tolist()[:10], "...")

📖 Leyendo primera hoja de: SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx


C:\Users\inter\AppData\Local\Temp\ipykernel_22404\1441234070.py:68: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_datos[cols_id] = df_datos[cols_id].replace(["", "None", "nan", "NaN"], np.nan).infer_objects(copy=False).ffill()


📖 Leyendo primera hoja de: SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx


C:\Users\inter\AppData\Local\Temp\ipykernel_22404\1441234070.py:68: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_datos[cols_id] = df_datos[cols_id].replace(["", "None", "nan", "NaN"], np.nan).infer_objects(copy=False).ffill()


📖 Leyendo primera hoja de: SALA DE DETENIDOS DÍA - 23022025 IT. AYALA.xlsx


...

C:\Users\inter\AppData\Local\Temp\ipykernel_22404\1441234070.py:68: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_datos[cols_id] = df_datos[cols_id].replace(["", "None", "nan", "NaN"], np.nan).infer_objects(copy=False).ffill()


📖 Leyendo primera hoja de: SALA DE DETENIDOS DÍA 31102024 SI. GARCIA.xlsx


C:\Users\inter\AppData\Local\Temp\ipykernel_22404\1441234070.py:68: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_datos[cols_id] = df_datos[cols_id].replace(["", "None", "nan", "NaN"], np.nan).infer_objects(copy=False).ffill()


📖 Leyendo primera hoja de: SALA DE DETENIDOS DÍA 31122024 IT. AYALA.xlsx

✅ Total de archivos procesados: 22
Columnas detectadas: ['RG', 'UNIDAD', 'UBICIACIÓN SALAS', 'TOTAL DE PERSONAS RETENIDAS EN INSTALACIONES POLICIALES Y URI', 'TOTAL DEL PERSONAL QUE CUSTODIA LOS RETENIDOS EN INSTALACIONES POLICIALES Y URI', 'RETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO', 'RETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS  DEPTO O METRO', 'RETENIDOS EN INSTALACIONES POLICIALES_SINDICADOS CON MAS DE 12 MESES', 'RETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS', 'RETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° CONDENADOS'] ...


C:\Users\inter\AppData\Local\Temp\ipykernel_22404\1441234070.py:68: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_datos[cols_id] = df_datos[cols_id].replace(["", "None", "nan", "NaN"], np.nan).infer_objects(copy=False).ffill()


In [23]:
print("--- 🔍 INICIO DEL DIAGNÓSTICO DE COLUMNAS ---")

archivos_con_problemas = 0

# Iteramos sobre la lista de DataFrames que ya tienes cargada
for i, df in enumerate(todos_los_datos):
    
    # 1. Evaluamos si el DataFrame tiene columnas con el mismo nombre
    if not df.columns.is_unique:
        archivos_con_problemas += 1
        
        # Intentamos obtener el nombre del archivo si guardamos el metadato, sino usamos el índice
        nombre_archivo = df['ARCHIVO_ORIGEN'].iloc[0] if 'ARCHIVO_ORIGEN' in df.columns else archivos[i]
        
        print(f"\n❌ ¡ALERTA! El archivo '{nombre_archivo}' tiene columnas DUPLICADAS.")
        
        # 2. Extraemos y mostramos exactamente cuáles son los nombres repetidos
        columnas_repetidas = df.columns[df.columns.duplicated()].unique().tolist()
        print(f"   -> Nombres repetidos: {columnas_repetidas}")
        
        # 3. Mostramos cuántas veces se repite cada una para entender la gravedad
        for col_rep in columnas_repetidas:
            cantidad = (df.columns == col_rep).sum()
            print(f"      La columna '{col_rep}' aparece {cantidad} veces.")

# 4. Conclusión del diagnóstico
if archivos_con_problemas == 0:
    print("\n✅ Ningún DataFrame individual tiene columnas duplicadas.")
    print("El problema debe estar en la diferencia de cantidad de columnas entre archivos.")
    
    # Si no hay duplicados, imprimimos la cantidad de columnas de cada uno para comparar
    print("\n--- Auditoría de dimensiones ---")
    for i, df in enumerate(todos_los_datos):
        nombre_archivo = df['ARCHIVO_ORIGEN'].iloc[0] if 'ARCHIVO_ORIGEN' in df.columns else archivos[i]
        print(f"Archivo {i+1}: {nombre_archivo} -> {len(df.columns)} columnas")
else:
    print(f"\n⚠️ RESUMEN: Se encontraron {archivos_con_problemas} archivos con nombres de columnas repetidos.")

--- 🔍 INICIO DEL DIAGNÓSTICO DE COLUMNAS ---

✅ Ningún DataFrame individual tiene columnas duplicadas.
El problema debe estar en la diferencia de cantidad de columnas entre archivos.

--- Auditoría de dimensiones ---
Archivo 1: SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx -> 57 columnas
Archivo 2: SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx -> 57 columnas
Archivo 3: SALA DE DETENIDOS DÍA - 23022025 IT. AYALA.xlsx -> 57 columnas
Archivo 4: SALA DE DETENIDOS DÍA - 28022025 IT. AYALA.xlsx -> 57 columnas
Archivo 5: SALA DE DETENIDOS DÍA - 29092024 IT. AYALA.xlsx -> 57 columnas
Archivo 6: SALA DE DETENIDOS DÍA - 31082024 IT. AYALA.xlsx -> 57 columnas
Archivo 7: SALA DE DETENIDOS DÍA 02032025 PT. DÁVILA.xlsx -> 61 columnas
Archivo 8: SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx -> 61 columnas
Archivo 9: SALA DE DETENIDOS DÍA 12032025 PT. DÁVILA.xlsx -> 61 columnas
Archivo 10: SALA DE DETENIDOS DÍA 17032025 IT. ACOSTA.xlsx -> 61 columnas
Archivo 11: SALA DE DETENIDOS DÍA 18092024 IT. 

In [ ]:
print("--- 🛠️ Reparando archivos afectados por columnas repetidas ---")

# 2. Aplicamos la reparación directamente a tu lista existente 'todos_los_datos'
for i in range(len(todos_los_datos)):
    todos_los_datos[i] = reparar_columnas_duplicadas(todos_los_datos[i])

print("✅ Todos los DataFrames tienen columnas únicas ahora.")

# 3. Concatenación Final
print("\n--- 🚀 Iniciando Concatenación de Big Data ---")
try:
    # ignore_index=True resetea los números de fila del 0 en adelante
    df_consolidado = pd.concat(todos_los_datos, axis=0, ignore_index=True)
    
    print("✅ ¡ÉXITO TOTAL! El error InvalidIndexError ha sido superado.")
    print(f"📊 Registros consolidados: {df_consolidado.shape[0]} filas.")
    print(f"📊 Total de columnas resultantes: {df_consolidado.shape[1]} columnas.")
    
    # Vista previa
    display(df_consolidado.head())
    
except Exception as e:
    print(f"❌ Error inesperado: {e}")

--- 🛠️ Reparando archivos afectados por columnas repetidas ---
✅ Todos los DataFrames tienen columnas únicas ahora.

--- 🚀 Iniciando Concatenación de Big Data ---
✅ ¡ÉXITO TOTAL! El error InvalidIndexError ha sido superado.
📊 Registros consolidados: 19598 filas.
📊 Total de columnas resultantes: 115 columnas.


,RG,UNIDAD,UBICIACIÓN SALAS,TOTAL DE PERSONAS RETENIDAS EN INSTALACIONES POLICIALES Y URI,TOTAL DEL PERSONAL QUE CUSTODIA LOS RETENIDOS EN INSTALACIONES POLICIALES Y URI,RETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO,RETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS DEPTO O METRO,RETENIDOS EN INSTALACIONES POLICIALES_SINDICADOS CON MAS DE 12 MESES,RETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS,RETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° CONDENADOS,...,"DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_AUX","DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL","DETENIDOS EN INSTALACIONES ""URI""_# DE EXTRANJEROS EN URI","DETENIDOS EN INSTALACIONES ""URI""_# DE EXTRANJEROS (VENEZOLANOS)","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° CONDENADOS_1","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° IMPUTADOS_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_M_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_F_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_LGBTI_1","DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIONES MÉDICAS (ESPECIALES CAPTURADOS VENEZOLANOS)"
0,NaN,MEBOG,CHAPINERO,NaN,NaN,21,296,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,MEBOG,SANTAFÉ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,MEBOG,SAN CRISTÓBAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,MEBOG,USME,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,MEBOG,TUNJUELITO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
# Unimos todos los DataFrames de la lista en uno solo
if todos_los_datos:
    
    df_final = pd.concat(todos_los_datos, ignore_index=True)
    print(f"✅ Dataset consolidado inicial con {df_final.shape[0]} filas.")
    
    # Creamos una condición para excluir lo que diga 'SUBTOTAL' en 'UNIDAD'
    condicion_unidad = ~df_final['UNIDAD'].str.contains('SUBTOTAL', case=False, na=False)
    
    # Creamos una condición para excluir lo que diga 'TOTAL GENERAL' en 'RG'
    condicion_rg = ~df_final['RG'].str.contains('TOTAL GENERAL', case=False, na=False)
    
    # 3. Aplicamos los filtros al DataFrame y reseteamos el índice
    df_final = df_final[condicion_unidad & condicion_rg].reset_index(drop=True)
    
    print(f"🧹 Dataset limpio (sin subtotales ni totales): {df_final.shape[0]} filas y {df_final.shape[1]} columnas.")
    
    # Vista previa para confirmar
    display(df_final.head())

else:
    print("No hay datos para consolidar.")

✅ Dataset consolidado inicial con 19598 filas.
🧹 Dataset limpio (sin subtotales ni totales): 19340 filas y 115 columnas.


,RG,UNIDAD,UBICIACIÓN SALAS,TOTAL DE PERSONAS RETENIDAS EN INSTALACIONES POLICIALES Y URI,TOTAL DEL PERSONAL QUE CUSTODIA LOS RETENIDOS EN INSTALACIONES POLICIALES Y URI,RETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO,RETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS DEPTO O METRO,RETENIDOS EN INSTALACIONES POLICIALES_SINDICADOS CON MAS DE 12 MESES,RETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS,RETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° CONDENADOS,...,"DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_AUX","DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL","DETENIDOS EN INSTALACIONES ""URI""_# DE EXTRANJEROS EN URI","DETENIDOS EN INSTALACIONES ""URI""_# DE EXTRANJEROS (VENEZOLANOS)","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° CONDENADOS_1","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° IMPUTADOS_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_M_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_F_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_LGBTI_1","DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIONES MÉDICAS (ESPECIALES CAPTURADOS VENEZOLANOS)"
0,NaN,MEBOG,CHAPINERO,NaN,NaN,21,296,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,MEBOG,SANTAFÉ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,MEBOG,SAN CRISTÓBAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,MEBOG,USME,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,MEBOG,TUNJUELITO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
#Mover 'ARCHIVO_ORIGEN' al principio de la tabla para fácil lectura
columnas = df_final.columns.tolist()

# Verificamos que la columna exista para evitar errores
if 'ARCHIVO_ORIGEN' in columnas:
    # Sacamos la columna de donde esté y la insertamos en la posición 0
    columnas.insert(0, columnas.pop(columnas.index('ARCHIVO_ORIGEN')))
    
    # Reasignamos el orden al DataFrame
    df_final = df_final[columnas]
    print("✅ Columna 'ARCHIVO_ORIGEN' movida a la primera posición.")
else:
    print("⚠️ No se encontró la columna 'ARCHIVO_ORIGEN'. Revisa la función de transformación.")

#¿Cuántos datos aportó cada archivo?
print("\n--- 📊 Contraste de datos por archivo original ---")
conteo_archivos = df_final['ARCHIVO_ORIGEN'].value_counts()
print(conteo_archivos)

# Vista previa para confirmar que ahora es la primera columna
display(df_final.head(10))

✅ Columna 'ARCHIVO_ORIGEN' movida a la primera posición.

--- 📊 Contraste de datos por archivo original ---
ARCHIVO_ORIGEN
SALA DE DETENIDOS DÍA 12032025 PT. DÁVILA.xlsx     1088
SALA DE DETENIDOS DÍA 31032025 PT. DÁVILA.xlsx     1088
SALA DE DETENIDOS DÍA 20042025 PT.DÁVILA.xlsx      1088
SALA DE DETENIDOS DÍA 22042025 IT. ACOSTA.xlsx     1088
SALA DE DETENIDOS DÍA 17032025 IT. ACOSTA.xlsx     1088
SALA DE DETENIDOS DÍA 02032025 PT. DÁVILA.xlsx     1085
SALA DE DETENIDOS DÍA 31102024 SI. GARCIA.xlsx     1085
SALA DE DETENIDOS DÍA 30092024 SI. GARCIA.xlsx     1085
SALA DE DETENIDOS DÍA 31012025 IT. ACOSTA.xlsx     1085
SALA DE DETENIDOS DÍA 30112024 PT. DÁVILA.xlsx     1085
SALA DE DETENIDOS DÍA 18092024 IT. ACOSTA.xlsx     1085
SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx     1085
SALA DE DETENIDOS DÍA 31072024 IT. ACOSTA.xlsx     1046
SALA DE DETENIDOS DÍA 30062024 IT. TOBÓN.xlsx      1046
SALA DE DETENIDOS DÍA 27022024 IT. TOBÓN.xlsx      1042
SALA DE DETENIDOS DÍA - 29092024 IT. 

,ARCHIVO_ORIGEN,RG,UNIDAD,UBICIACIÓN SALAS,TOTAL DE PERSONAS RETENIDAS EN INSTALACIONES POLICIALES Y URI,TOTAL DEL PERSONAL QUE CUSTODIA LOS RETENIDOS EN INSTALACIONES POLICIALES Y URI,RETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO,RETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS DEPTO O METRO,RETENIDOS EN INSTALACIONES POLICIALES_SINDICADOS CON MAS DE 12 MESES,RETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS,...,"DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_AUX","DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL","DETENIDOS EN INSTALACIONES ""URI""_# DE EXTRANJEROS EN URI","DETENIDOS EN INSTALACIONES ""URI""_# DE EXTRANJEROS (VENEZOLANOS)","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° CONDENADOS_1","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° IMPUTADOS_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_M_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_F_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_LGBTI_1","DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIONES MÉDICAS (ESPECIALES CAPTURADOS VENEZOLANOS)"
0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,CHAPINERO,NaN,NaN,21,296,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,SANTAFÉ,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,SAN CRISTÓBAL,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,USME,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,TUNJUELITO,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,BOSA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,BOSA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,KENNEDY,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,FONTIBÓN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,MEBOG,ENGATIVÁ,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
